In [1]:
import os

In [2]:
%pwd

'/workspaces/MLOps-Car-Price-Prediction-System/notebooks'

In [3]:
os.chdir("../")

In [4]:
%pwd

'/workspaces/MLOps-Car-Price-Prediction-System'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_file: str
    train_data_file_path: str
    test_data_file_path: str
    preprocessor_obj_file_path: str

In [6]:
from Car_Price_Prediction_System.constants import *
from Car_Price_Prediction_System.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(self, config_filepath=Config_Filepath, params_filepath=Params_Filepath, schema_filepath=Schema_Filepath):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir = config.root_dir,
            data_file = config.data_file,
            train_data_file_path = config.train_data_file_path,
            test_data_file_path = config.test_data_file_path,
            preprocessor_obj_file_path = config.preprocessor_obj_file_path
        )

        return data_transformation_config

In [ ]:
from Car_Price_Prediction_System import logger
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import joblib

In [ ]:
class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config

    def create_preprocessor(self, X_train):
        logger.info("Starting Data Preprocessing")
        categorical_columns = X_train.select_dtypes(include=['object']).columns.to_list()
        numerical_columns = X_train.select_dtypes(include=['number']).columns.to_list()
    
        preprocessor = ColumnTransformer(transformers=[
            ('cat', OneHotEncoder(handle_unknown="ignore", drop='first'), categorical_columns),
            ('num', StandardScaler(), numerical_columns)
        ])
    
        preprocessor.fit(X_train)
        logger.info("Completed Data Preprocessing")
            
        logger.info("Saved Preprocessor Object")
        joblib.dump(preprocessor, self.config.preprocessor_obj_file_path)

        return preprocessor
    
    def save_train_test_data(self):
        logger.info("Loading Dataset")
        data = pd.read_csv(self.config.data_file)

        X = data.drop(['Car_Name', 'Selling_Price'], axis=1)
        y = data['Selling_Price']

        logger.info("Splitting Dataset")
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        preprocessor = self.create_preprocessor(X_train)

        train_df = pd.DataFrame(preprocessor.transform(X_train))
        train_df["Selling_Price"] = y_train.reset_index(drop=True)

        test_df = pd.DataFrame(preprocessor.transform(X_test))
        test_df["Selling_Price"] = y_test.reset_index(drop=True)

        train_df.to_csv(self.config.train_data_file_path, index=False)
        test_df.to_csv(self.config.test_data_file_path, index=False)
        logger.info("Training And Testing Data Saved Successfully")

In [ ]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.save_train_test_data()
except Exception as e:
    raise e

[2026-07-22 07:08:48,405] : INFO : common : Created File: <_io.TextIOWrapper name='config/config.yaml' mode='r' encoding='UTF-8'>
[2026-07-22 07:08:48,483] : INFO : common : Created File: <_io.TextIOWrapper name='config/params.yaml' mode='r' encoding='UTF-8'>
[2026-07-22 07:08:48,498] : INFO : common : Created File: <_io.TextIOWrapper name='config/schema.yaml' mode='r' encoding='UTF-8'>
[2026-07-22 07:08:48,504] : INFO : common : Created Directory: artifacts
[2026-07-22 07:08:48,521] : INFO : common : Created Directory: artifacts/data_transformation/data
[2026-07-22 07:08:48,523] : INFO : 2455847686 : Loading Dataset
[2026-07-22 07:08:48,564] : INFO : 2455847686 : Splitting Dataset
[2026-07-22 07:08:48,582] : INFO : 2455847686 : Training And Testing Data Saved Successfully
[2026-07-22 07:08:48,584] : INFO : 2455847686 : Starting Data Preprocessing
[2026-07-22 07:08:49,437] : INFO : 2455847686 : Completed Data Preprocessing
[2026-07-22 07:08:49,438] : INFO : 2455847686 : Saved Preproces